# Test the library, one feature at a time

This notebook checks the whole library against the **live** IdroGEO service.
Each cell uses one feature, prints what came back, and records a check.
The last cell prints a table: ✅ passed, ❌ failed. If anything fails, that
cell raises an error, so `make test-notebooks` fails too.

It downloads only small files (a few hundred kB) and takes about a minute.
Nothing needs an account or a key.

In [1]:
# pip install "ispra-idrogeo[geo,mcp,warehouse]"
import json
import tempfile
from pathlib import Path

import ispra_idrogeo
from ispra_idrogeo import (
    LANDSLIDE_FIELDS,
    IdroGEO,
    IdrogeoError,
    decode,
    event_filter,
    landslide_filter,
    opendata,
)

results: list[tuple[str, bool, str]] = []


def check(name: str, ok: bool, detail: object = "") -> None:
    """Record one check and say how it went."""
    results.append((name, bool(ok), str(detail)))
    print(("✅" if ok else "❌"), name, "·", detail)


idrogeo = IdroGEO()  # one client for the whole notebook
work = Path(tempfile.mkdtemp(prefix="idrogeo-test-"))  # downloads go here, not in your cache
print("ispra-idrogeo", ispra_idrogeo.__version__)

ispra-idrogeo 0.7.0


## 1. Download a file, then ask again

`fetch` downloads and unpacks a file. The second time it asks the server
"has it changed?" and gets a short `304` answer instead of the file.

In [2]:
folder = await idrogeo.fetch(opendata("dgpv_poly", "molise"), work)
files = sorted(p.suffix for p in folder.iterdir())
check("fetch unpacks a shapefile", ".shp" in files, files)

again = await idrogeo.download(opendata("dgpv_poly", "molise"), work)
check("a second download costs a 304", not again.changed, f"changed={again.changed}")

✅ fetch unpacks a shapefile · ['.cst', '.dbf', '.prj', '.shp', '.shx']


✅ a second download costs a 304 · changed=False


## 2. Dated landslide events

The `eventi` file has the landslide events ISPRA dated. Most dates give only
the day: the time is `00:00` or missing.

In [3]:
events_folder = await idrogeo.fetch(opendata("eventi", fmt="geojson"), work)
[geojson] = events_folder.glob("*.json")
events = [f["properties"] for f in json.loads(geojson.read_text())["features"]]
recent = [e for e in events if (e["data_certa"] or "") >= "2018"]
day_only = [e for e in recent if e["data_certa"][11:16] in ("", "00:00")]
check("events have a date field", "data_certa" in events[0], f"{len(events):,} events")
check(
    "since 2018, most events give only the day",
    len(day_only) > len(recent) / 2,
    f"{len(day_only):,} of {len(recent):,} without a time",
)

✅ events have a date field · 1,808 events
✅ since 2018, most events give only the day · 1,466 of 1,710 without a time


## 3. Risk figures of a town

`summary("pir", "comuni", code)` gives hazard and exposure for one
municipality, by ISTAT code. Bari is 72006.

In [4]:
bari = await idrogeo.summary("pir", "comuni", 72006)
check("hazard and exposure of Bari", bari["nome"] == "Bari", f"{bari['popidp2_p']} % in flood P2")

sarno = await idrogeo.summary("iffi", "comuni", 65135)
check("IFFI counts of Sarno", sarno["frane_totali"] > 0, f"{sarno['frane_totali']} landslides")

✅ hazard and exposure of Bari · 1.278 % in flood P2
✅ IFFI counts of Sarno · 53 landslides


## 4. Search landslides

A search with no `select` asks for `LANDSLIDE_FIELDS`. Before 0.7.1 this
search failed with HTTP 400 (issue #9).

In [5]:
rows = await idrogeo.search_landslides(limit=3)
check("a search without select works (#9)", set(LANDSLIDE_FIELDS) <= rows[0].keys(), len(rows))

sinkholes = await idrogeo.search_landslides(landslide_filter(movement=6), limit=5)
check("filter by type of movement", all(r["tipo_movimento"] == 6 for r in sinkholes), "sinkholes")

record = await idrogeo.landslide(sinkholes[0]["id_frana"])
check("one landslide in full", len(record) > 100, f"{len(record)} fields")
try:
    await idrogeo.landslide("0000000000")
    check("an unknown landslide raises IdrogeoError", False)
except IdrogeoError as exc:
    check("an unknown landslide raises IdrogeoError", True, exc)

✅ a search without select works (#9) · 3


✅ filter by type of movement · sinkholes
✅ one landslide in full · 142 fields
✅ an unknown landslide raises IdrogeoError · landslide 0000000000 is unknown or not public (HTTP 401)


## 5. Every landslide of a town

`iter_landslides_in` reads the whole inventory with a few fields and keeps
the landslides of one area. It takes about 20 seconds for Italy.

In [6]:
in_sarno = [f async for f in idrogeo.iter_landslides_in(municipality=65135)]
check(
    "the landslides of Sarno match the IFFI count",
    len(in_sarno) == sarno["frane_totali"],
    f"{len(in_sarno)} vs {sarno['frane_totali']}",
)

✅ the landslides of Sarno match the IFFI count · 53 vs 53


## 6. Events by area and date

In [7]:
liguria = await idrogeo.search_events(event_filter(region=7), limit=5)
check("events of Liguria", all(e["regione"] == 7 for e in liguria), len(liguria))

✅ events of Liguria · 5


## 7. Turn codes into words

The data stores codes. `decode` adds a `<field>_label` next to each coded
field.

In [8]:
labels = await idrogeo.code_labels(["tipo_movimento"], lang="en")
named = decode(record, labels)
check("decode adds labels", named["tipo_movimento_label"], named["tipo_movimento_label"])

✅ decode adds labels · sinkhole


## 8. Hazard at a point, and landslides nearby

Both ask IdroGEO's public map service: no big download.

In [9]:
here = await idrogeo.hazard_at(40.829, 14.617)  # the hills above Sarno
check("landslide hazard above Sarno is P4", here.landslide == "P4", here.municipality_name)

venice = await idrogeo.hazard_at(45.438, 12.330)
check("Venice is in flood hazard P3", venice.flood == "P3", venice.flood_scenarios)

near = await idrogeo.landslides_near(40.838, 14.630, radius_m=500)
distances = [f["distance_m"] for f in near]
check("landslides near a point, nearest first", near and distances == sorted(distances), len(near))

✅ landslide hazard above Sarno is P4 · Sarno


✅ Venice is in flood hazard P3 · ('P3', 'P2', 'P1')


✅ landslides near a point, nearest first · 1


## 9. Monitoring sites and reference lists

In [10]:
sites = await idrogeo.sites()
check("monitoring sites", len(sites) > 500, f"{len(sites):,} sites")
check("one site by its uid", (await idrogeo.site("SIPIE001"))["uid"] == "SIPIE001", "SIPIE001")

units = await idrogeo.lookup("formazioni", "argille", limit=2)
check("search the geological units", len(units) > 0, units[0].get("nome_unita"))

✅ monitoring sites · 1,063 sites
✅ one site by its uid · SIPIE001


✅ search the geological units · ARGILLE E ARENARIE GLAUCONITICHE DI CATENANUOVA


## 10. Read a download as a GeoDataFrame

Needs the `[geo]` extra. The coordinates are in EPSG:32632.

In [11]:
from ispra_idrogeo.geo import read

frame = read(folder)
check("read() gives a GeoDataFrame in EPSG:32632", frame.crs.to_epsg() == 32632, len(frame))

✅ read() gives a GeoDataFrame in EPSG:32632 · 3


## 11. A local warehouse, in SQL

Needs the `[warehouse]` extra. DuckDB downloads its `spatial` extension the
first time.

In [12]:
from ispra_idrogeo.warehouse import Warehouse

with Warehouse(work / "test.duckdb") as warehouse:
    await warehouse.sync(idrogeo, ["dgpv_poly"], area="molise")
    [row] = warehouse.query("SELECT count(*) AS n FROM dgpv_poly")
check("the warehouse loads and answers SQL", row["n"] == len(frame), row)

✅ the warehouse loads and answers SQL · {'n': 3}


## 12. The MCP server, without a network port

Needs the `[mcp]` extra. The same tools an AI agent sees, called directly.

In [13]:
from ispra_idrogeo.mcp_server import server

mcp = server(idrogeo)
tools = {t.name for t in await mcp.list_tools()}
check("the MCP server lists its tools", {"hazard_at", "find_area"} <= tools, f"{len(tools)} tools")

found = await mcp.call_tool("find_area", {"name": "forli"})
check("find_area ignores accents", "Forl" in found.content[0].text, "forli → Forlì")

✅ the MCP server lists its tools · 16 tools


HTTP Request: GET https://idrogeo.isprambiente.it/api/iffi/comuni "HTTP/1.1 200 OK"


✅ find_area ignores accents · forli → Forlì


## Result

In [14]:
await idrogeo.aclose()
failed = [name for name, ok, _ in results if not ok]
for name, ok, detail in results:
    print("✅" if ok else "❌", f"{name:<52}", detail[:60])
print(f"\n{len(results) - len(failed)} of {len(results)} checks passed")
assert not failed, f"failed: {failed}"

✅ fetch unpacks a shapefile                            ['.cst', '.dbf', '.prj', '.shp', '.shx']
✅ a second download costs a 304                        changed=False
✅ events have a date field                             1,808 events
✅ since 2018, most events give only the day            1,466 of 1,710 without a time
✅ hazard and exposure of Bari                          1.278 % in flood P2
✅ IFFI counts of Sarno                                 53 landslides
✅ a search without select works (#9)                   3
✅ filter by type of movement                           sinkholes
✅ one landslide in full                                142 fields
✅ an unknown landslide raises IdrogeoError             landslide 0000000000 is unknown or not public (HTTP 401)
✅ the landslides of Sarno match the IFFI count         53 vs 53
✅ events of Liguria                                    5
✅ decode adds labels                                   sinkhole
✅ landslide hazard above Sarno is P4                 